# BCSS-WSSS - Stage 1 ResNet38d Multi-Label Classification & CAM Training
## Production-Grade Colab Pro Notebook: 23,422 BCSS-WSSS Training Patches
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khoinguyen59/medical-wsss/blob/master/notebooks/01_Train_Stage1_Classifier.ipynb)


* **Task:** Train Stage 1 ResNet-38d dilated backbone (105M parameters) on 23,422 weakly-supervised breast cancer histopathology patches (BCSS-WSSS).
* **Recommended Hardware:** NVIDIA L4 GPU (24GB VRAM, High-RAM) or Tesla T4 (16GB VRAM).
* **Precision:** Mixed Precision BF16 (L4) or FP16 (T4) with PyTorch AMP.
* **Loss Formulation:** Multi-Label Soft Margin Loss with Progressive Dropout Attention (PDA) activation for extensive Class Activation Map (CAM) coverage.
* **Checkpoints:** Direct high-speed NVMe SSD storage on Colab VM disk (`/content/checkpoints/`) with instant one-click download (Zero Google Drive dependency).


---
## Step 1: Configure High-Speed Colab Local Storage (/content/checkpoints/) - Zero Drive Required
Configures high-speed local NVMe SSD storage at `/content/checkpoints/` directly on the Colab VM disk.


In [ ]:
import os
import sys
from pathlib import Path

# Check if running inside Google Colab
IN_COLAB = "google.colab" in sys.modules

# Pure Colab Local NVMe Storage (Pure GitHub + Colab, Zero Drive Popup)
if IN_COLAB:
    CHECKPOINT_DIR = Path("/content/checkpoints")
    print("[STORAGE] Using high-speed Colab VM local disk at /content/checkpoints")
else:
    CHECKPOINT_DIR = Path("./checkpoints")

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
STAGE1_LATEST_CKPT = CHECKPOINT_DIR / "stage1_latest.pth"
STAGE1_BEST_CKPT = CHECKPOINT_DIR / "stage1_retrained_best.pth"
print(f"[CONFIG] Checkpoint directory: {CHECKPOINT_DIR.resolve()}")
print(f"[CONFIG] Latest checkpoint target: {STAGE1_LATEST_CKPT}")
print(f"[CONFIG] Best checkpoint target: {STAGE1_BEST_CKPT}")


---
## Step 2: Auto-Discover Codebase & Initialize Reproducible Environment
Searches for `src` directory across Google Drive, local paths, or falls back to GitHub clone.


In [ ]:
import os
import sys
from pathlib import Path
import random
import subprocess
import zipfile
import numpy as np
import torch

# Pure GitHub Codebase Discovery
CANDIDATES = [
    Path("/content/medical_wsss_repo"),
    Path("/content/medical-wsss"),
    Path.cwd(),
    Path.cwd().parent,
]

CODE_ROOT = next((str(p.resolve()) for p in CANDIDATES if (p / "src" / "models").is_dir()), None)
if CODE_ROOT is None:
    print("[GIT] Cloning official repository https://github.com/khoinguyen59/medical-wsss.git ...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/khoinguyen59/medical-wsss.git", "/content/medical_wsss_repo"], check=False)
    if Path("/content/medical_wsss_repo/src/models").is_dir():
        CODE_ROOT = str(Path("/content/medical_wsss_repo").resolve())
    else:
        CODE_ROOT = str(Path.cwd().resolve())

if CODE_ROOT in sys.path:
    sys.path.remove(CODE_ROOT)
sys.path.insert(0, CODE_ROOT)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    major_cc = torch.cuda.get_device_capability()[0]
    AMP_DTYPE = torch.bfloat16 if major_cc >= 8 else torch.float16
else:
    AMP_DTYPE = torch.float32
print(f"[READY] Code root: {CODE_ROOT}")
print(f"[READY] Compute Device: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"[READY] Mixed Precision AMP Dtype: {AMP_DTYPE}")


---
## Step 3: Resilient Dataset Download & Ingestion (23,422 Patches)
Downloads and extracts `BCSS-WSSS.zip` via HuggingFace mirror with high-speed mirror.


In [ ]:
import os
import sys
import zipfile
import urllib.request
from pathlib import Path

# Dataset locations
IN_COLAB = 'google.colab' in sys.modules if 'IN_COLAB' not in globals() else IN_COLAB
DATASET_DIR = Path('/content/BCSS-WSSS') if IN_COLAB else Path('./data/BCSS-WSSS')
DATASET_ZIP = Path('/content/BCSS-WSSS.zip') if IN_COLAB else Path('./data/BCSS-WSSS.zip')
DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)

HF_URL = 'https://huggingface.co/datasets/nttduc/wsss-pathology/resolve/main/BCSS-WSSS.zip'
GDRIVE_ID = '1sCK8Cqsf4yCHzzIx98vitocES2L_tkjn'

# Robust image presence check (searches recursively for any PNG patches)
def count_patches(folder):
    p = Path(folder)
    return len(list(p.rglob('*.png'))) if p.is_dir() else 0

# Check both author 'training/' and 'train/' folders for existing patches
total_train_patches = count_patches(DATASET_DIR / 'training') + count_patches(DATASET_DIR / 'train')
if total_train_patches < 100:
    if not DATASET_ZIP.is_file() or DATASET_ZIP.stat().st_size < 1000000:
        print(f'[DOWNLOAD] Downloading BCSS-WSSS dataset from HuggingFace mirror: {HF_URL}')
        try:
            urllib.request.urlretrieve(HF_URL, str(DATASET_ZIP))
            print(f'[DOWNLOAD] Download complete ({DATASET_ZIP.stat().st_size / (1024**3):.2f} GB)')
        except Exception as e:
            print(f'[WARN] HuggingFace mirror failed: {e}. Attempting gdown from Drive...')
            import subprocess
            subprocess.run(['pip', 'install', '-q', 'gdown'], check=False)
            subprocess.run(['gdown', '--id', GDRIVE_ID, '-O', str(DATASET_ZIP)], check=False)

    if DATASET_ZIP.is_file() and DATASET_ZIP.stat().st_size > 1000000:
        print(f'[EXTRACT] Extracting {DATASET_ZIP} to {DATASET_DIR} ...')
        with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
            zf.extractall(str(DATASET_DIR))
        print('[EXTRACT] Extraction finished successfully.')

# Auto-resolve TRAIN_DIR and VAL dirs supporting both author 'training/' and 'train/' conventions
def resolve_split_path(base_dir, split='train', sub=None):
    if split in ('train', 'training'):
        split_aliases = ['training', 'train']
    elif split in ('val', 'valid', 'validation'):
        split_aliases = ['val', 'valid', 'validation']
    else:
        split_aliases = [split]
    candidates = []
    for alias in split_aliases:
        if sub:
            candidates.extend([
                base_dir / alias / sub,
                base_dir / 'BCSS-WSSS' / alias / sub,
                Path('/content/BCSS-WSSS') / alias / sub,
                Path('/content') / alias / sub,
                Path('./data/BCSS-WSSS') / alias / sub,
                Path('./data') / alias / sub,
            ])
        else:
            candidates.extend([
                base_dir / alias,
                base_dir / alias / 'img',
                base_dir / 'BCSS-WSSS' / alias,
                base_dir / 'BCSS-WSSS' / alias / 'img',
                Path('/content/BCSS-WSSS') / alias,
                Path('/content/BCSS-WSSS') / alias / 'img',
                Path('/content') / alias,
                Path('/content') / alias / 'img',
                Path('./data/BCSS-WSSS') / alias,
                Path('./data') / alias,
            ])
    for c in candidates:
        if c.is_dir() and len(list(c.glob('*.png'))) > 0:
            return c
    for c in candidates:
        if c.is_dir() and len(list(c.rglob('*.png'))) > 0:
            return c
    default_alias = split_aliases[0]
    return (base_dir / default_alias / sub) if sub else (base_dir / default_alias)

TRAIN_DIR = resolve_split_path(DATASET_DIR, 'train')
VAL_IMG_DIR = resolve_split_path(DATASET_DIR, 'val', 'img')
VAL_MASK_DIR = resolve_split_path(DATASET_DIR, 'val', 'mask')
train_patches = list(TRAIN_DIR.glob('*.png')) if TRAIN_DIR.is_dir() else []
if len(train_patches) == 0 and TRAIN_DIR.is_dir():
    train_patches = list(TRAIN_DIR.rglob('*.png'))
val_patches = list(VAL_IMG_DIR.glob('*.png')) if VAL_IMG_DIR.is_dir() else []
if len(val_patches) == 0 and VAL_IMG_DIR.is_dir():
    val_patches = list(VAL_IMG_DIR.rglob('*.png'))
print(f'[DATASET] Active training directory: {TRAIN_DIR} (Patches: {len(train_patches):,})')
print(f'[DATASET] Active validation images: {VAL_IMG_DIR} (Patches: {len(val_patches):,})')
print(f'[DATASET] Active validation masks: {VAL_MASK_DIR}')


---
## Step 4: High-Performance Multi-Label Dataset & DataLoader Pipeline
Extracts weak multi-label diagnostic targets `[TUM STR LYM NEC]` from filenames with patient-level anti-leakage verification.


In [ ]:
import re
import random
from collections import defaultdict
from PIL import Image
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from src.training.dataset_loader import extract_patient_id, load_bcss_patient_splits

# Baseline Parity input normalization: 'zero_one' matches repos/WSSS-Tissue/1_train_stage1.py
# and domain-pretrained ResNet-38d checkpoints; metadata 'input_norm' is persisted for Stage 2.
STAGE1_INPUT_NORM = 'zero_one'  # Options: 'zero_one' (Recommended, paper parity) | 'imagenet'

def _sample_patient_stratified(files, max_samples, seed=42):
    if max_samples is None or len(files) <= max_samples:
        return list(files)
    rng = random.Random(seed)
    by_pid = defaultdict(list)
    for f in files:
        by_pid[extract_patient_id(f.name)].append(f)
    pids = sorted(by_pid.keys())
    for pid in pids:
        rng.shuffle(by_pid[pid])
    selected = []
    idx = 0
    while len(selected) < max_samples:
        progressed = False
        for pid in pids:
            if idx < len(by_pid[pid]):
                selected.append(by_pid[pid][idx])
                progressed = True
                if len(selected) >= max_samples:
                    break
        if not progressed:
            break
        idx += 1
    return selected

class BCSSStage1Dataset(Dataset):
    """BCSS-WSSS Stage 1 multi-label histopathology dataset with patient-level anti-leakage."""
    def __init__(self, img_dir, transform=None, limit=None, patient_ids=None):
        self.img_dir = Path(img_dir)
        self.transform = transform
        files = list(self.img_dir.glob('*.png'))
        if len(files) == 0 and (self.img_dir / 'img').is_dir():
            files = list((self.img_dir / 'img').glob('*.png'))
        if len(files) == 0 and self.img_dir.is_dir():
            files = list(self.img_dir.rglob('*.png'))
        if len(files) == 0:
            raise FileNotFoundError(f'[STAGE1 ERROR] No patches found in {self.img_dir}. Please ensure dataset extraction completed successfully!')
        files = sorted(files)
        if patient_ids is not None:
            allowed_pids = {str(p).upper() for p in patient_ids}
            filtered = [f for f in files if extract_patient_id(f.name) in allowed_pids]
            if len(filtered) > 0:
                files = filtered
        self.files = _sample_patient_stratified(files, limit, seed=42) if (limit is not None and limit > 0) else files
        self.pattern = re.compile(r'\[([0-9\s,]+)\]')
        print(f'[DATASET] Stage 1 Train ingested {len(self.files):,} patches from {self.img_dir}')

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        filepath = self.files[idx]
        image = Image.open(filepath).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        match = self.pattern.search(filepath.name)
        if match:
            raw_str = match.group(1).replace(',', ' ').strip()
            digits = [float(c) for c in raw_str if c.isdigit()]
            if len(digits) == 4:
                label = torch.tensor(digits, dtype=torch.float32)
            else:
                tokens = raw_str.split()
                label = torch.tensor([float(x) for x in tokens], dtype=torch.float32) if len(tokens) == 4 else torch.zeros(4, dtype=torch.float32)
        else:
            label = torch.zeros(4, dtype=torch.float32)
        return filepath.name, image, label

class BCSSStage1ValDataset(Dataset):
    """Independent Stage 1 validation dataset deriving true multi-hot labels and GT masks from val/mask."""
    def __init__(self, img_dir, mask_dir, transform=None, limit=None, patient_ids=None):
        self.img_dir = Path(img_dir)
        self.mask_dir = Path(mask_dir) if mask_dir is not None else None
        self.transform = transform
        files = sorted(list(self.img_dir.glob('*.png'))) if self.img_dir.is_dir() else []
        if len(files) == 0 and self.img_dir.is_dir():
            files = sorted(list(self.img_dir.rglob('*.png')))
        if patient_ids is not None:
            allowed_pids = {str(p).upper() for p in patient_ids}
            filtered = [f for f in files if extract_patient_id(f.name) in allowed_pids]
            if len(filtered) > 0:
                files = filtered
        self.files = _sample_patient_stratified(files, limit, seed=42) if (limit is not None and limit > 0) else files
        uniq_pids = len({extract_patient_id(f.name) for f in self.files})
        print(f'[DATASET] Stage 1 Val ingested {len(self.files):,} patches across {uniq_pids} unique patients (Anti-Leakage Verified).')

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        filepath = self.files[idx]
        image = Image.open(filepath).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        mask_path = (self.mask_dir / filepath.name) if (self.mask_dir is not None and self.mask_dir.is_dir()) else None
        if mask_path is not None and mask_path.is_file():
            m_img = Image.open(mask_path)
            if m_img.mode != 'P':
                m_img = m_img.convert('L')
            gt_arr = np.array(m_img.resize((224, 224), Image.NEAREST), dtype=np.int64)
            if gt_arr.ndim == 3:
                gt_arr = gt_arr[:, :, 0]
            label = torch.tensor([1.0 if np.any(gt_arr == c) else 0.0 for c in range(4)], dtype=torch.float32)
        else:
            gt_arr = np.full((224, 224), 255, dtype=np.int64)
            label = torch.zeros(4, dtype=torch.float32)
        return filepath.name, image, label, torch.from_numpy(gt_arr).long()

# Data augmentation pipeline aligned with STAGE1_INPUT_NORM
_train_ops = [
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.03),
    transforms.ToTensor(),
]
_val_ops = [
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
]
if STAGE1_INPUT_NORM == 'imagenet':
    _norm_op = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    _train_ops.append(_norm_op)
    _val_ops.append(_norm_op)
train_transform = transforms.Compose(_train_ops)
val_transform = transforms.Compose(_val_ops)

BATCH_SIZE = 32
NUM_WORKERS = 4
DRY_RUN = False if 'DRY_RUN' not in globals() else DRY_RUN  # Set to True for fast smoke-test
patient_splits = load_bcss_patient_splits()
train_patient_ids = patient_splits.get('train', [])
val_patient_ids = patient_splits.get('val', [])
assert set(train_patient_ids).isdisjoint(set(val_patient_ids)), 'Invariant 08 violation: train/val patient overlap!'

train_dataset = BCSSStage1Dataset(
    TRAIN_DIR,
    transform=train_transform,
    limit=64 if DRY_RUN else None,
    patient_ids=train_patient_ids,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=min(BATCH_SIZE, max(1, len(train_dataset))),
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=(len(train_dataset) >= BATCH_SIZE),
)
val_dataset = BCSSStage1ValDataset(
    VAL_IMG_DIR,
    VAL_MASK_DIR,
    transform=val_transform,
    limit=32 if DRY_RUN else None,
    patient_ids=val_patient_ids,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=min(BATCH_SIZE, max(1, len(val_dataset))),
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
) if len(val_dataset) > 0 else None
print(f'[DATALOADER] Train batches/epoch: {len(train_loader)} (BS={BATCH_SIZE}, Samples={len(train_dataset):,}, norm={STAGE1_INPUT_NORM})')
print(f'[DATALOADER] Val batches/epoch: {len(val_loader) if val_loader else 0} (Samples={len(val_dataset):,})')


---
## Step 5: Instantiate Stage 1 ResNet38d Net_CAM & Progressive Dropout Attention (PDA)
Loads the official 105M-parameter ResNet-38d architecture with dilated convolutions and Net_CAM projection.


In [ ]:
import os
import sys
from pathlib import Path
import torch
import numpy as np

# Self-healing fallback definitions for DEVICE and AMP_DTYPE
if 'DEVICE' not in globals():
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if 'AMP_DTYPE' not in globals():
    AMP_DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
if 'DRY_RUN' not in globals():
    DRY_RUN = False
if 'STAGE1_INPUT_NORM' not in globals():
    STAGE1_INPUT_NORM = 'zero_one'
if 'CHECKPOINT_DIR' not in globals():
    CHECKPOINT_DIR = Path('/content/checkpoints') if ('google.colab' in sys.modules) else Path('./checkpoints')

try:
    from src.models.wsss_inference_methods import WSSSTissueStage1NetCAM
    print('[IMPORT] Successfully imported WSSSTissueStage1NetCAM from src.models')
except ImportError:
    from src.colab_pkg.model_loader import WSSSTissueStage1NetCAM
    print('[IMPORT] Imported WSSSTissueStage1NetCAM from src.colab_pkg')

from src.training.train_stage1 import (
    AsymmetricLoss,
    AttentionModule,
    build_optimizer_and_scheduler,
    compute_multilabel_validation_metrics,
    load_pretrained_resnet38d_weights,
)

# Instantiate Stage 1 ResNet-38d Net_CAM
model = WSSSTissueStage1NetCAM(num_classes=4)

# Rule 17 Pretrained Weight Initialization Gate (NEVER silently train 105M ResNet-38d from scratch)
# Supports: local ilsvrc-cls_rna-a1_cls1000_ep-0001.params/.pth OR cross-organ LUAD histopathology ResNet-38d backbone (fc8 reset)
PRETRAINED_WEIGHTS_PATH = None  # Optional explicit path to ImageNet .params or .pth file
AUTO_DOWNLOAD_DOMAIN_PRETRAINED = True
ALLOW_RANDOM_INIT = DRY_RUN  # Strictly False for real training unless explicitly overridden

PRETRAINED_INFO = load_pretrained_resnet38d_weights(
    model,
    pretrained_path=PRETRAINED_WEIGHTS_PATH,
    allow_random_init=ALLOW_RANDOM_INIT,
    reset_classifier_head=True,
    auto_download_domain_pretrained=AUTO_DOWNLOAD_DOMAIN_PRETRAINED,
    cache_dir=CHECKPOINT_DIR,
)

model = model.to(DEVICE)
attention_module = AttentionModule().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'[MODEL] Stage 1 ResNet38d Net_CAM: {total_params:,} params | Pretrained: {PRETRAINED_INFO["source"]} ({PRETRAINED_INFO["loaded_layers"]}/{PRETRAINED_INFO["total_backbone_layers"]} layers)')

# Baseline Parity Hyperparameters (Rule 17 Gradient-Step & LR Scaling Parity vs paper BS=20, 20 ep = 23,420 steps)
MAX_EPOCHS = 3 if DRY_RUN else 25
LEARNING_RATE = 1.4e-4
WEIGHT_DECAY = 1e-4
LOSS_TYPE = 'soft_margin'  # Baseline Parity default: 'soft_margin' (MultiLabelSoftMarginLoss) | Optional: 'asl'

optimizer, scheduler = build_optimizer_and_scheduler(
    model, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, epochs=MAX_EPOCHS
)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda' and AMP_DTYPE == torch.float16))
if LOSS_TYPE == 'asl':
    # Calibrated gentle ASL (gamma_neg=2.0, clip=0.02) to prevent minority-class threshold collapse
    criterion = AsymmetricLoss(gamma_neg=2.0, gamma_pos=0.0, clip=0.02)
else:
    criterion = torch.nn.MultiLabelSoftMarginLoss()

_steps_per_ep = len(train_loader) if 'train_loader' in globals() else 0
print(f'[PARITY] Loss={type(criterion).__name__} (LOSS_TYPE={LOSS_TYPE}) | norm={STAGE1_INPUT_NORM}')
print(f'[PARITY] Epochs={MAX_EPOCHS} | LR={LEARNING_RATE} | Steps/Epoch={_steps_per_ep} | Total Steps={_steps_per_ep * MAX_EPOCHS:,} (Paper ref: 23,420)')

def forward_stage1_with_pda(model, images, enable_pda=False, gamma=0.7):
    # Forward pass through dilated backbone; keep PDA + 4096-ch head in FP32 with autocast disabled
    conv6 = model.forward_backbone(images)  # (B, 4096, 28, 28)
    with torch.amp.autocast(device_type=conv6.device.type, enabled=False):
        conv6_f = conv6.float()
        if enable_pda:
            conv6_f = attention_module(conv6_f, model.fc8.weight.float(), gamma=gamma)
        x = model.dropout7(conv6_f)
        x = model.pool(x)
        logits = torch.nn.functional.conv2d(x, model.fc8.weight.float()).view(x.size(0), -1)
        probs = torch.sigmoid(logits)
        return logits, probs


---
## Step 6: Automatic Resume from Previous Epoch Checkpoint
Checks if a previous training run was interrupted, restoring optimizer and scheduler state seamlessly.


In [ ]:
import torch
if 'DEVICE' not in globals():
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

start_epoch = 0
best_val_score = -1.0
best_val_metric_name = 'val_gated_pm_miou'
best_acc = 0.0
pda_gamma = 1.0
history = {
    'epoch': [], 'loss': [], 'em': [], 'acc': [], 'lr': [],
    'val_loss': [], 'val_mAP': [], 'val_auroc': [], 'val_best_f1': [],
    'val_gated_pm_miou': [], 'val_gated_pm_mdice': [],
}

if STAGE1_LATEST_CKPT.is_file():
    print(f'[RESUME] Found existing checkpoint: {STAGE1_LATEST_CKPT}')
    try:
        ckpt = torch.load(STAGE1_LATEST_CKPT, map_location=DEVICE, weights_only=False)
        state_dict = ckpt.get('state_dict', ckpt.get('model', ckpt))
        model.load_state_dict(state_dict, strict=False)
        if 'optimizer' in ckpt:
            optimizer.load_state_dict(ckpt['optimizer'])
        if 'scheduler' in ckpt:
            scheduler.load_state_dict(ckpt['scheduler'])
        start_epoch = ckpt.get('epoch', 0) + 1
        best_val_score = ckpt.get('best_val_score', ckpt.get('val_gated_pm_miou', -1.0))
        best_val_metric_name = ckpt.get('best_val_metric', 'val_gated_pm_miou')
        best_acc = ckpt.get('best_acc', ckpt.get('macro_f1', 0.0))
        pda_gamma = ckpt.get('pda_gamma', 1.0)
        loaded_hist = ckpt.get('history', {})
        for k in history:
            if k in loaded_hist:
                history[k] = loaded_hist[k]
        print(f'[RESUME] Resuming training from epoch {start_epoch}/{MAX_EPOCHS} (Best {best_val_metric_name}: {best_val_score:.4f})')
    except Exception as e:
        print(f'[WARN] Failed to load checkpoint: {e}. Starting fresh training.')
else:
    print('[INIT] No previous checkpoint found. Starting Stage 1 training from initialized backbone.')


---
## Step 7: Multi-Epoch Training Loop with Interactive tqdm Progress Tracking
Trains ResNet38d with Progressive Dropout Attention (PDA) enabled after epoch 2 and automatic epoch checkpointing.


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from tqdm.auto import tqdm

# Self-healing fallback definitions for DEVICE and AMP_DTYPE
if 'DEVICE' not in globals():
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if 'AMP_DTYPE' not in globals():
    AMP_DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
if 'pda_gamma' not in globals():
    pda_gamma = 1.0
if 'STAGE1_INPUT_NORM' not in globals():
    STAGE1_INPUT_NORM = 'zero_one'
if 'LOSS_TYPE' not in globals():
    LOSS_TYPE = 'soft_margin'
if 'PRETRAINED_INFO' not in globals():
    PRETRAINED_INFO = {'source': 'unknown', 'loaded_layers': 0}

# Rule 17 Anti-Blind-Training Gate: require independent validation loop on non-dry-run
if not DRY_RUN and ('val_loader' not in globals() or val_loader is None or len(val_loader) == 0):
    raise RuntimeError('[STAGE1 ERROR] Independent val_loader is missing! Refusing blind training without validation (Rule 17).')

@torch.no_grad()
def evaluate_stage1_validation(model_eval, loader_eval):
    model_eval.eval()
    val_loss_sum = 0.0
    val_n = 0
    all_probs, all_targets = [], []
    pm_inter = np.zeros(4, dtype=np.float64)
    pm_union = np.zeros(4, dtype=np.float64)
    pm_dnum = np.zeros(4, dtype=np.float64)
    pm_dden = np.zeros(4, dtype=np.float64)
    has_masks = False

    for _, imgs_v, lbls_v, gts_v in loader_eval:
        imgs_v = imgs_v.to(DEVICE, non_blocking=True)
        lbls_v = lbls_v.to(DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=(DEVICE.type == 'cuda')):
            conv6_v = model_eval.forward_backbone(imgs_v)
        with torch.amp.autocast('cuda', enabled=False):
            conv6_f = conv6_v.float()
            fc_w = model_eval.fc8.weight.float()
            logits_v = F.conv2d(model_eval.pool(conv6_f), fc_w).view(conv6_f.size(0), -1)
            probs_v = torch.sigmoid(logits_v)
            loss_v = criterion(logits_v, lbls_v)
            val_loss_sum += float(loss_v.item()) * imgs_v.size(0)
            val_n += imgs_v.size(0)
            all_probs.append(probs_v.cpu().numpy())
            all_targets.append(lbls_v.cpu().numpy())

            gt_np = gts_v.numpy()
            valid_roi = (gt_np >= 0) & (gt_np < 4)
            if np.any(valid_roi):
                has_masks = True
                cams_v = F.relu(F.conv2d(conv6_f, fc_w))
                cam_up = F.interpolate(cams_v, size=gt_np.shape[-2:], mode='bilinear', align_corners=False)
                gate = lbls_v.float().view(-1, 4, 1, 1)
                cam_g = cam_up * gate
                b_sz = cam_g.size(0)
                cmin = cam_g.view(b_sz, 4, -1).min(dim=-1)[0].view(b_sz, 4, 1, 1)
                cmax = cam_g.view(b_sz, 4, -1).max(dim=-1)[0].view(b_sz, 4, 1, 1)
                cam_norm = ((cam_g - cmin) / (cmax - cmin).clamp(min=1e-7)) * gate
                cam_masked = torch.where(gate > 0.5, cam_norm, torch.full_like(cam_norm, -1.0))
                pred_pm = torch.argmax(cam_masked, dim=1).cpu().numpy()
                for c in range(4):
                    pc = (pred_pm == c) & valid_roi
                    gc = (gt_np == c) & valid_roi
                    tp = float(np.logical_and(pc, gc).sum())
                    pm_inter[c] += tp
                    pm_union[c] += float(np.logical_or(pc, gc).sum())
                    pm_dnum[c] += 2.0 * tp
                    pm_dden[c] += float(pc.sum() + gc.sum())

    y_prob = np.concatenate(all_probs, axis=0) if all_probs else np.zeros((0, 4), dtype=np.float32)
    y_true = np.concatenate(all_targets, axis=0) if all_targets else np.zeros((0, 4), dtype=np.float32)
    metrics_v = compute_multilabel_validation_metrics(y_true, y_prob)
    metrics_v['val_loss'] = val_loss_sum / max(1, val_n)
    if has_masks:
        ious_c = [float(pm_inter[c] / pm_union[c]) if pm_union[c] > 0 else 0.0 for c in range(4)]
        dices_c = [float(pm_dnum[c] / pm_dden[c]) if pm_dden[c] > 0 else 0.0 for c in range(4)]
        active_i = [ious_c[c] for c in range(4) if pm_union[c] > 0]
        active_d = [dices_c[c] for c in range(4) if pm_dden[c] > 0]
        metrics_v['val_gated_pm_miou'] = float(np.mean(active_i)) if active_i else 0.0
        metrics_v['val_gated_pm_mdice'] = float(np.mean(active_d)) if active_d else 0.0
        metrics_v['val_pm_ious'] = [round(x, 4) for x in ious_c]
    return metrics_v

epochs_without_improve = 0
for epoch in range(start_epoch, MAX_EPOCHS):
    model.train()
    running_loss = 0.0
    running_em = 0
    running_acc = 0.0
    total_samples = 0
    skipped_batches = 0

    # Progressive Dropout Attention (PDA) activation condition (Han et al. 2022)
    enable_pda = (epoch >= 2)

    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1:02d}/{MAX_EPOCHS:02d} [PDA={enable_pda}, gamma={pda_gamma:.2f}]', unit='batch', mininterval=10.0)
    for step, (filenames, images, labels) in enumerate(pbar):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        batch_n = labels.size(0)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=(DEVICE.type == 'cuda')):
            logits, probs = forward_stage1_with_pda(model, images, enable_pda=enable_pda, gamma=pda_gamma)
            loss = criterion(logits, labels)

        if not torch.isfinite(loss):
            skipped_batches += 1
            continue

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_samples += batch_n
        running_loss += float(loss.detach().item()) * batch_n
        with torch.no_grad():
            pred_bin = probs.detach() > 0.5
            gt_bin = labels.detach() > 0.5
            inter = (pred_bin & gt_bin).sum(dim=1).float()
            union = (pred_bin | gt_bin).sum(dim=1).float()
            acc_vec = torch.where(union > 0, inter / union.clamp(min=1.0), torch.ones_like(inter))
            running_em += int((pred_bin == gt_bin).all(dim=1).sum().item())
            running_acc += float(acc_vec.sum().item())

        if (step + 1) % 100 == 0 or (step + 1) == len(train_loader):
            current_vram = f'{torch.cuda.memory_allocated() / (1024**3):.1f}GB' if torch.cuda.is_available() else 'N/A'
            pbar.set_postfix({
                'loss': f'{running_loss / max(1, total_samples):.4f}',
                'acc': f'{running_acc / max(1, total_samples):.4f}',
                'lr': f'{optimizer.param_groups[0]["lr"]:.2e}',
                'vram': current_vram,
            })

    scheduler.step()
    if enable_pda and pda_gamma > 0.65:
        pda_gamma = max(0.65, pda_gamma * 0.98)
    epoch_loss = running_loss / max(1, total_samples)
    epoch_em = running_em / max(1, total_samples)
    epoch_acc = running_acc / max(1, total_samples)

    # Independent Validation Loop (Patient-Separated, Rule 17)
    val_metrics = evaluate_stage1_validation(model, val_loader) if ('val_loader' in globals() and val_loader is not None) else {}
    val_loss = float(val_metrics.get('val_loss', 0.0))
    val_map = float(val_metrics.get('val_mAP', 0.0))
    val_auroc = float(val_metrics.get('val_auroc', 0.0))
    val_best_f1 = float(val_metrics.get('val_best_f1', 0.0))
    val_pm_miou = float(val_metrics.get('val_gated_pm_miou', 0.0))
    val_pm_mdice = float(val_metrics.get('val_gated_pm_mdice', 0.0))

    history['epoch'].append(epoch + 1)
    history['loss'].append(round(epoch_loss, 5))
    history['em'].append(round(epoch_em, 5))
    history['acc'].append(round(epoch_acc, 5))
    history['lr'].append(optimizer.param_groups[0]['lr'])
    history['val_loss'].append(round(val_loss, 5))
    history['val_mAP'].append(round(val_map, 4))
    history['val_auroc'].append(round(val_auroc, 4))
    history['val_best_f1'].append(round(val_best_f1, 4))
    history['val_gated_pm_miou'].append(round(val_pm_miou, 4))
    history['val_gated_pm_mdice'].append(round(val_pm_mdice, 4))

    print(
        f'[EPOCH {epoch+1:02d}] Train Loss={epoch_loss:.4f} EM={epoch_em:.4f} Acc={epoch_acc:.4f} | '
        f'Val mAP={val_map:.4f} AUROC={val_auroc:.4f} BestF1={val_best_f1:.4f} '
        f'GatedPM_mIoU={val_pm_miou:.4f} (Dice={val_pm_mdice:.4f}, IoUs={val_metrics.get("val_pm_ious", [])})',
        flush=True,
    )

    if total_samples > 0 and np.isfinite(epoch_loss):
        # Select best checkpoint strictly by independent validation metric (NEVER train acc)
        if 'val_gated_pm_miou' in val_metrics:
            best_val_metric_name = 'val_gated_pm_miou'
            current_val_score = val_pm_miou
        else:
            best_val_metric_name = 'val_mAP'
            current_val_score = val_map

        improved = (current_val_score >= best_val_score)
        if improved:
            best_val_score = current_val_score
            best_acc = max(best_acc, val_best_f1)
            epochs_without_improve = 0
        else:
            epochs_without_improve += 1

        latest_state = {
            'epoch': epoch,
            'state_dict': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'input_norm': STAGE1_INPUT_NORM,
            'loss_type': LOSS_TYPE,
            'pretrained_source': PRETRAINED_INFO.get('source', 'unknown'),
            'pretrained_loaded_layers': PRETRAINED_INFO.get('loaded_layers', 0),
            'best_val_score': best_val_score,
            'best_val_metric': best_val_metric_name,
            'best_acc': best_acc,
            'macro_f1': val_best_f1,
            'val_mAP': val_map,
            'val_auroc': val_auroc,
            'val_best_f1': val_best_f1,
            'val_gated_pm_miou': val_pm_miou,
            'val_gated_pm_mdice': val_pm_mdice,
            'val_metrics': val_metrics,
            'pda_gamma': pda_gamma,
            'history': history,
        }
        torch.save(latest_state, STAGE1_LATEST_CKPT)
        print(f'[CHECKPOINT] Saved epoch {epoch+1} to {STAGE1_LATEST_CKPT} (input_norm={STAGE1_INPUT_NORM})', flush=True)

        if improved or not STAGE1_BEST_CKPT.is_file():
            best_state = {k: v for k, v in latest_state.items() if k not in ('optimizer', 'scheduler')}
            torch.save(best_state, STAGE1_BEST_CKPT)
            print(f'[CHECKPOINT] >>> New best {best_val_metric_name}={best_val_score:.4f} (val_mAP={val_map:.4f})! Saved compact best ckpt to {STAGE1_BEST_CKPT}', flush=True)

        # Early Regression Alarm (Rule 17 Section 2)
        if not DRY_RUN and epoch >= 4:
            if val_pm_miou < 0.45 and best_val_metric_name == 'val_gated_pm_miou':
                print(f'[EARLY REGRESSION ALARM] Epoch {epoch+1}: val_gated_pm_miou={val_pm_miou:.4f} is far below Author Baseline (0.7146). Inspect LR/pretrained weights!', flush=True)
            if epochs_without_improve >= 5:
                print(f'[EARLY REGRESSION ALARM] No validation improvement for {epochs_without_improve} consecutive epochs (best {best_val_metric_name}={best_val_score:.4f}).', flush=True)
    else:
        raise RuntimeError(f'Non-finite epoch loss at epoch {epoch+1}: {epoch_loss}')


---
## Step 8: Plot Training Curves & Export Deliverables ZIP
Generates 300 DPI loss/accuracy curves, exports JSON metrics, and triggers automatic browser download.


In [ ]:
import os
import sys
import json
import shutil
import zipfile
from pathlib import Path
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules if 'IN_COLAB' not in globals() else IN_COLAB
CHECKPOINT_DIR = Path('/content/checkpoints') if IN_COLAB else Path('./checkpoints')
STAGE1_BEST_CKPT = CHECKPOINT_DIR / 'stage1_retrained_best.pth' if 'STAGE1_BEST_CKPT' not in globals() else STAGE1_BEST_CKPT
STAGE1_LATEST_CKPT = CHECKPOINT_DIR / 'stage1_latest.pth' if 'STAGE1_LATEST_CKPT' not in globals() else STAGE1_LATEST_CKPT
STAGE1_INPUT_NORM = globals().get('STAGE1_INPUT_NORM', 'zero_one')

OUTPUT_DIR = Path('/content/stage1_results') if IN_COLAB else Path('./stage1_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Bundle compact best checkpoint into output package (keep resume-only latest ckpt on disk to avoid 3x ZIP bloat)
if STAGE1_BEST_CKPT.is_file():
    shutil.copy(STAGE1_BEST_CKPT, OUTPUT_DIR / STAGE1_BEST_CKPT.name)
    print(f'[PACKAGE] Bundled best checkpoint {STAGE1_BEST_CKPT.name} ({STAGE1_BEST_CKPT.stat().st_size / (1024**2):.1f} MB)')
if STAGE1_LATEST_CKPT.is_file():
    print(f'[RESUME] Latest optimizer checkpoint retained on disk at {STAGE1_LATEST_CKPT} ({STAGE1_LATEST_CKPT.stat().st_size / (1024**2):.1f} MB)')

# Plot training loss and independent validation curves
if history.get('epoch'):
    fig, ax1 = plt.subplots(figsize=(9, 5), dpi=300)
    ax1.set_xlabel('Epoch', fontweight='bold')
    ax1.set_ylabel('Loss', color='tab:red', fontweight='bold')
    ax1.plot(history['epoch'], history['loss'], color='tab:red', marker='o', label='Train Loss')
    if history.get('val_loss'):
        ax1.plot(history['epoch'], history['val_loss'], color='tab:orange', linestyle='--', marker='.', label='Val Loss')
    ax1.tick_params(axis='y', labelcolor='tab:red')
    ax1.grid(True, linestyle='--', alpha=0.5)

    ax2 = ax1.twinx()
    ax2.set_ylabel('Validation Metrics (mAP / AUROC / PM mIoU)', color='tab:blue', fontweight='bold')
    if history.get('val_mAP'):
        ax2.plot(history['epoch'], history['val_mAP'], color='tab:blue', marker='s', label='Val mAP')
    if history.get('val_auroc'):
        ax2.plot(history['epoch'], history['val_auroc'], color='tab:purple', marker='d', label='Val AUROC')
    if history.get('val_gated_pm_miou'):
        ax2.plot(history['epoch'], history['val_gated_pm_miou'], color='tab:green', marker='^', label='Val Gated PM mIoU')
    ax2.tick_params(axis='y', labelcolor='tab:blue')

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right', fontsize=8)
    plt.title('BCSS-WSSS Stage 1 ResNet38d Training & Independent Validation Curves', fontweight='bold')
    fig.tight_layout()
    plot_path = OUTPUT_DIR / 'stage1_training_curves.png'
    plt.savefig(plot_path, dpi=300)
    plt.close()
    print(f'[VISUALIZATION] Saved training plot to {plot_path}')

# Save metrics JSON
metrics_path = OUTPUT_DIR / 'STAGE1_TRAINING_METRICS.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump({
        'best_val_metric': globals().get('best_val_metric_name', 'val_gated_pm_miou'),
        'best_val_score': globals().get('best_val_score', 0.0),
        'best_accuracy': globals().get('best_acc', 0.0),
        'input_norm': STAGE1_INPUT_NORM,
        'loss_type': globals().get('LOSS_TYPE', 'soft_margin'),
        'pretrained_info': globals().get('PRETRAINED_INFO', {}),
        'history': history,
    }, f, indent=2)
print(f'[METRICS] Saved metrics JSON to {metrics_path}')

# Compress results and trigger download in Colab
ZIP_EXPORT = Path('/content/SE374_Stage1_Training_Results.zip') if IN_COLAB else Path('./SE374_Stage1_Training_Results.zip')
with zipfile.ZipFile(ZIP_EXPORT, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in OUTPUT_DIR.glob('*'):
        if f.is_file():
            zf.write(f, arcname=f.name)
print(f'[PACKAGE] Compressed results to {ZIP_EXPORT} ({ZIP_EXPORT.stat().st_size / (1024**2):.1f} MB)')

if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(ZIP_EXPORT))
        print('[DOWNLOAD] Triggered browser download of results ZIP.')
    except Exception as e:
        print(f'[NOTICE] Browser download skipped: {e}')
